<a href="https://colab.research.google.com/github/ZakParkinson04/Using-Chest-X-rays-to-diagnose-diseases-and-anomalies-an-explainable-AI-Model-for-medical-Diagnosis/blob/main/Dissertation_Chest_XAI_Diagnosis_Support_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
Project_Direct = "/content/drive/MyDrive/Dissertation Chest XAI Diagnosis Support"

In [ ]:
#Creating project folders
import os

folders = [
    "data/raw",
    "data/processed",
    "outputs/models/densenet",
    "outputs/models/resnet",
    "outputs/predictions",
    "outputs/figures",
    "outputs/metrics",
    "outputs/heatmaps",
    "notebooks"
]

for folder in folders:
  os.makedirs(os.path.join(Project_Direct, folder), exist_ok=True)

print("Project folders created.")

Project folders created.


In [ ]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 61.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=c30708e6e7ebf38fa303d0183f8c9a30e4d5d7f82821a61c53e332361af58693
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

import matplotlib.pyplot as plt

print("PyTorch version: ", torch.__version__)
print("Torchvision version: ", torchvision.__version__)
print("GPU available: ", torch.cuda.is_available())


PyTorch version:  2.10.0+cu128
Torchvision version:  0.25.0+cu128
GPU available:  True


In [ ]:
Data_Direct = os.path.join(Project_Direct, "data/raw")
Output_Direct = os.path.join(Project_Direct, "outputs")

Image_Size = 224
Batch_Size = 32
Num_Epochs = 10
Learning_Rate = 1e-4

Main_Model = "densenet121"
Bench_Model = "resnet50"
Pretrained = True



In [ ]:
Train_transform = transforms.Compose([
    transforms.Resize((Image_Size, Image_Size)),
    transforms.Grayscale(num_output_channels = 3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

Test_transform = transforms.Compose([
    transforms.Resize((Image_Size, Image_Size)),
    transforms.Grayscale(num_output_channels = 3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [ ]:
from torchvision.datasets import FakeData

NUM_CLASSES = 2
CLASS_NAMES = ["normal", "disease"]

dummy_dataset = FakeData(
    size=200,
    image_size = (3, Image_Size, Image_Size),
    num_classes = NUM_CLASSES,
    transform = Test_transform
)

In [ ]:
Train_Size = int(0.7 * len(dummy_dataset))
Val_Size = int(0.15 * len(dummy_dataset))
Test_Size = len(dummy_dataset) - Train_Size - Val_Size

Train_Dataset, Val_Dataset, Test_Dataset = random_split(
    dummy_dataset,
    [Train_Size, Val_Size, Test_Size]
)

In [ ]:
Traning_Load = DataLoader(Train_Dataset, Batch_Size, shuffle=True)
Validation_Load = DataLoader(Val_Dataset, Batch_Size, shuffle=False)
Testing_Load = DataLoader(Test_Dataset, Batch_Size, shuffle=False)

print("Dummy Dataset Loades.")
print("Train Size: ", len(Train_Dataset))
print("Validation Size: ", len(Val_Dataset))
print("Test Size: ", len(Test_Dataset))


Dummy Dataset Loades.
Train Size:  140
Validation Size:  30
Test Size:  30


In [ ]:
def densenet121(Num_Classes, Pretrained = True):
  weights = models.DenseNet121_Weights.DEFAULT if Pretrained else None
  model = models.densenet121(weights=weights)

  in_features = model.classifier.in_features
  model.classifier = nn.Linear(in_features, Num_Classes)

  return model

In [ ]:
def resnet50(Num_Classes, Pretrained = True):
  weights = models.Resnet50_Weights.DEFAULT if Pretrained else None
  model = models.resnet50(weights=weights)

  in_features = model.fc.in_features
  model = nn.Linear(in_features, Num_Classes)

  return model

In [ ]:
def Train_One_Epoch(model, Train_Loader, Criterion, Optimizer, Device):
  model.train()

  Running_Loss= 0.0
  Correct = 0
  Total = 0

  for Images, Labels in tqdm(Train_Loader):
    Images = Images.to(Device)
    Labels = Labels.to(Device)

    Optimizer.zero_grad()

    Outputs = model(Images)
    Loss = Criterion(Outputs, Labels)
    Loss.backward()

    Optimizer.step()
    Running_Loss += Loss.item()

    _, Predicted = torch.max(Outputs, 1)
    Total += Labels.size(0)
    Correct += (Predicted == Labels).sum().item()

  Epoch_Loss = Running_Loss / len(Train_Loader)
  Epoch_Accuracy = Correct / Total

  return Epoch_Loss, Epoch_Accuracy

  print("Epoch Loss:", Epoch_Loss)
  print("Epoch Accuracy:", Epoch_Accuracy)

In [ ]:
def Validattion(model, Train_Loader, Criterion, Device):
  model.eval()

  Running_Loss = 0.0
  Correct = 0
  Total = 0
  with torch.no_grad():
    for Images, Labels in tqdm(Train_Loader):
      Images = Images.to(Device)
      Labels = Labels.to(Device)


      Outputs = model(Images)
      Loss = Criterion(Outputs, Labels)

      Running_Loss += Loss.item()

      _, Predicted = torch.max(Outputs, 1)
      Total += Labels.size(0)
      Correct += (Predicted == Labels).sum().item()

  Val_Loss = Running_Loss / len(Train_Loader)
  Val_Accuracy = Correct / Total

  return Val_Loss, Val_Accuracy

  print("Validation Loss:", Val_Loss)
  print("Validation Accuracy:", Val_Accuracy)

In [ ]:
Device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = densenet121(
    Num_Classes = NUM_CLASSES,
    Pretrained = Pretrained
).to(Device)

Criterion = nn.CrossEntropyLoss()
Optimizer = optim.Adam(model.parameters(), lr=Learning_Rate)

Test_Epochs = 1
for Epoch in range(Test_Epochs):
    print(f"Epoch {Epoch + 1}/{Test_Epochs}")
    Train_Epoch_Loss, Train_Epoch_Accuracy = Train_One_Epoch(
        model,
        Traning_Load,
        Criterion,
        Optimizer,
        Device
    )

    Val_Loss, Val_Acc = Validattion(
        model,
        Validation_Load,
        Criterion,
        Device
    )

    print(f"Train Loss: {Train_Epoch_Loss:.4f}, Train Accuracy: {Train_Epoch_Accuracy:.4f}")
    print(f"Validation Loss: {Val_Loss:.4f}, Validation Accuracy: {Val_Acc:.4f}")

Epoch 1/1


100%|██████████| 1/1 [00:00<00:00,  5.05it/s]

Train Loss: 0.7236, Train Accuracy: 0.4857
Validation Loss: 0.7217, Validation Accuracy: 0.5667
